# Demo: Build an AI Container Vulnerability Assessment with Trivy

This notebook walks through a Trivy-style vulnerability assessment for a containerized drone imagery inference service. The same workflow is available as `scripts/run_trivy_assessment_demo.py` for command-line execution.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from trivy_assessment_utils import (
    build_assessment,
    extract_misconfigurations,
    extract_vulnerabilities,
    format_priority_table,
    load_json,
    render_markdown_report,
    summarize_sbom,
    validate_trivy_report,
    write_csv,
    write_json,
)

report_path = ROOT / "data" / "sample_trivy_report.json"
sbom_path = ROOT / "data" / "sample_sbom.cdx.json"
results_dir = ROOT / "results"

## 1. Validate the Trivy JSON format

Trivy JSON reports organize findings under a `Results` list. Each result usually maps to a scan target, such as OS packages, Python packages, or Dockerfile misconfigurations.

In [ ]:
report = load_json(report_path)
validate_trivy_report(report)

## 2. Extract vulnerabilities and misconfigurations

AI workloads inherit risk from OS packages, Python libraries, model-serving dependencies, image parsers, and container settings.

In [ ]:
vulnerabilities = extract_vulnerabilities(report)
misconfigurations = extract_misconfigurations(report)

len(vulnerabilities), len(misconfigurations)

## 3. Connect the report to the SBOM

Trivy can emit SBOM output that documents the software components present in the image. This inventory supports downstream AI security reviews and exception tracking.

In [ ]:
sbom = load_json(sbom_path)
sbom_components = summarize_sbom(sbom)
sbom_components[:5]

## 4. Prioritize findings by operational exposure

Severity matters, but it is not enough. For this inference service, vulnerabilities near attacker-supplied images, multipart upload handling, network services, and root container execution receive higher priority.

In [ ]:
assessment = build_assessment(report, vulnerabilities, misconfigurations, sbom_components)
print(format_priority_table(assessment["top_operational_risks"]))

## 5. Export final-project style artifacts

In [ ]:
results_dir.mkdir(parents=True, exist_ok=True)
write_json(assessment, results_dir / "ai_container_vulnerability_assessment.json")
write_csv(assessment["top_operational_risks"], results_dir / "vulnerability_log.csv")
write_csv(sbom_components, results_dir / "sbom_component_summary.csv")
(results_dir / "supply_chain_analysis.md").write_text(render_markdown_report(assessment), encoding="utf-8")
sorted(path.name for path in results_dir.iterdir())